# Real-Time ASL Fingerspelling Recognition Pipeline

This notebook handles downloading the dataset, extracting landmarks, normalizing them, training the model, and exporting it in one go.

## 1. Setup & Download Dataset

In [ ]:
# @title
!pip install kaggle mediapipe==0.10.35 opencv-python pandas tqdm scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0


In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the target Google Drive directory
DRIVE_DIR = '/content/drive/MyDrive/asl_project_files'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"Google Drive directory ready at: {DRIVE_DIR}")

Mounted at /content/drive
Google Drive directory ready at: /content/drive/MyDrive/asl_project_files


In [ ]:
import os

# Pre-flight checklist to verify paths and cache status
print("=== ASL PIPELINE PRE-FLIGHT CHECK ===\n")

paths_to_check = {
    "Google Drive Directory": DRIVE_DIR,
    "Dataset Zip (.zip)": os.path.join(DRIVE_DIR, 'asl-alphabet.zip'),
    "Raw Landmarks (.csv)": os.path.join(DRIVE_DIR, 'landmarks_raw.csv'),
    "Normalized Features (.csv)": os.path.join(DRIVE_DIR, 'landmarks_normalized.csv'),
    "Trained Model (.pkl)": os.path.join(DRIVE_DIR, 'best_asl_model.pkl')
}

all_clear = True
for label, path in paths_to_check.items():
    exists = os.path.exists(path)
    status = "✅ AVAILABLE (Will load instantly)" if exists else "❌ MISSING (Will download/process)"
    print(f"{label}:\n  Path: {path}\n  Status: {status}\n")

print("=====================================")
print("If the Drive directory exists, you are ready to run the cells sequentially!")

=== ASL PIPELINE PRE-FLIGHT CHECK ===

Google Drive Directory:
  Path: /content/drive/MyDrive/asl_project_files
  Status: ✅ AVAILABLE (Will load instantly)

Dataset Zip (.zip):
  Path: /content/drive/MyDrive/asl_project_files/asl-alphabet.zip
  Status: ❌ MISSING (Will download/process)

Raw Landmarks (.csv):
  Path: /content/drive/MyDrive/asl_project_files/landmarks_raw.csv
  Status: ❌ MISSING (Will download/process)

Normalized Features (.csv):
  Path: /content/drive/MyDrive/asl_project_files/landmarks_normalized.csv
  Status: ❌ MISSING (Will download/process)

Trained Model (.pkl):
  Path: /content/drive/MyDrive/asl_project_files/best_asl_model.pkl
  Status: ❌ MISSING (Will download/process)

If the Drive directory exists, you are ready to run the cells sequentially!


In [ ]:
import os

dataset_zip_path = os.path.join(DRIVE_DIR, 'asl-alphabet.zip')

# Check if dataset already exists in Google Drive to avoid downloading it again
if os.path.exists(dataset_zip_path):
    print("Dataset zip found in Google Drive! Using the cached file.")
else:
    print("Dataset not found in Drive. Uploading kaggle.json and downloading dataset...")
    from google.colab import files
    print('Please upload your kaggle.json file:')
    uploaded = files.upload()

    !mkdir -p ~/.kaggle
    !cp kaggle.json ~/.kaggle/
    !chmod 600 ~/.kaggle/kaggle.json
    !kaggle datasets download -d grassknoted/asl-alphabet

    # Move the downloaded zip to Google Drive for future runs
    !mv asl-alphabet.zip "{dataset_zip_path}"
    print(f"Saved dataset zip to Drive: {dataset_zip_path}")

# Unzip dataset locally to the fast Colab disk for processing
!mkdir -p data/raw
if not os.path.exists('data/raw/asl-alphabet'):
    print("Unzipping dataset to local runtime disk...")
    !unzip -q "{dataset_zip_path}" -d data/raw/asl-alphabet
    print("Unzipping complete.")
else:
    print("Dataset already unzipped locally.")

Dataset not found in Drive. Uploading kaggle.json and downloading dataset...
Please upload your kaggle.json file:


Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/grassknoted/asl-alphabet
License(s): GPL-2.0
100% 1.03G/1.03G [00:11<00:00, 95.9MB/s]

Saved dataset zip to Drive: /content/drive/MyDrive/asl_project_files/asl-alphabet.zip
Unzipping dataset to local runtime disk...
Unzipping complete.


## 2. Extract Hand Landmarks using MediaPipe

In [ ]:
import cv2
import mediapipe as mp
import os
import glob
import pandas as pd
from tqdm import tqdm
import urllib.request

drive_raw_csv = os.path.join(DRIVE_DIR, 'landmarks_raw.csv')
os.makedirs('data/processed', exist_ok=True)

# Skip landmark extraction if raw landmarks already exist in Google Drive
if os.path.exists(drive_raw_csv):
    print(f"Raw landmarks found in Google Drive! Copying to local runtime...")
    df = pd.read_csv(drive_raw_csv)
    df.to_csv('data/processed/landmarks_raw.csv', index=False)
    print(f"Loaded {len(df)} records from Google Drive. Skipping MediaPipe extraction.")
else:
    print("Downloading hand_landmarker.task model...")
    url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
    if not os.path.exists("hand_landmarker.task"):
        urllib.request.urlretrieve(url, "hand_landmarker.task")

    BaseOptions = mp.tasks.BaseOptions
    HandLandmarker = mp.tasks.vision.HandLandmarker
    HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
    VisionRunningMode = mp.tasks.vision.RunningMode

    options = HandLandmarkerOptions(
        base_options=BaseOptions(model_asset_path='hand_landmarker.task'),
        running_mode=VisionRunningMode.IMAGE,
        num_hands=1,
        min_hand_detection_confidence=0.5)

    dataset_path = 'data/raw/asl-alphabet/asl_alphabet_train/asl_alphabet_train'
    classes = os.listdir(dataset_path)

    data = []
    failed = []

    print("Extracting landmarks using new Tasks API (this may take a while)...")
    with HandLandmarker.create_from_options(options) as landmarker:
        for cls in tqdm(classes):
            img_paths = glob.glob(os.path.join(dataset_path, cls, '*.jpg'))
            # Uncomment to prototype fast:
            # img_paths = img_paths[:200]

            for img_path in img_paths:
                img = cv2.imread(img_path)
                if img is None: continue

                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=img_rgb)

                detection_result = landmarker.detect(mp_image)

                if len(detection_result.hand_landmarks) == 1:
                    landmarks = detection_result.hand_landmarks[0]
                    handedness = detection_result.handedness[0][0].category_name
                    row = {'class': cls, 'handedness': handedness, 'image_path': img_path}
                    for i, lm in enumerate(landmarks):
                        row[f'x_{i}'] = lm.x
                        row[f'y_{i}'] = lm.y
                        row[f'z_{i}'] = lm.z
                    data.append(row)
                else:
                    failed.append({'path': img_path})

    df = pd.DataFrame(data)
    df.to_csv('data/processed/landmarks_raw.csv', index=False)
    df.to_csv(drive_raw_csv, index=False)
    print(f"\nSuccessfully processed {len(df)} images and backed up raw landmarks to Google Drive.")

Extracting landmarks using new Tasks API (this may take a while)...


100%|██████████| 29/29 [58:58<00:00, 122.02s/it]



Successfully processed 63580 images and backed up raw landmarks to Google Drive.


## 3. Normalize Landmarks
Translate to wrist origin, scale by hand size, and mirror left hands to look like right hands.

In [ ]:
import numpy as np
import pandas as pd
import os

drive_norm_csv = os.path.join(DRIVE_DIR, 'landmarks_normalized.csv')

# Skip normalization if normalized file is already cached in Google Drive
if os.path.exists(drive_norm_csv):
    print("Normalized landmarks found in Google Drive! Loading directly...")
    df_final = pd.read_csv(drive_norm_csv)
    df_final.to_csv('data/processed/landmarks_normalized.csv', index=False)
    print("Done loading normalized features.")
else:
    df = pd.read_csv('data/processed/landmarks_raw.csv')

    def normalize_landmarks(row):
        x = np.array([row[f'x_{i}'] for i in range(21)])
        y = np.array([row[f'y_{i}'] for i in range(21)])
        z = np.array([row[f'z_{i}'] for i in range(21)])

        if row['handedness'] == 'Left':
            x = 1.0 - x

        x = x - x[0]
        y = y - y[0]
        z = z - z[0]

        scale = np.sqrt(x[9]**2 + y[9]**2 + z[9]**2)
        if scale > 0:
            x, y, z = x / scale, y / scale, z / scale

        res = {}
        for i in range(21):
            res[f'norm_x_{i}'] = x[i]
            res[f'norm_y_{i}'] = y[i]
            res[f'norm_z_{i}'] = z[i]
        return pd.Series(res)

    print("Normalizing features...")
    normalized = df.apply(normalize_landmarks, axis=1)
    df_final = pd.concat([df[['class', 'image_path']], normalized], axis=1)
    df_final.to_csv('data/processed/landmarks_normalized.csv', index=False)
    df_final.to_csv(drive_norm_csv, index=False)
    print("Normalization complete. Backed up normalized landmarks to Google Drive.")

Normalizing features...
Normalization complete. Backed up normalized landmarks to Google Drive.


## 4. Train Models
Train a Random Forest model as our baseline classifier.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib
import os

drive_model_path = os.path.join(DRIVE_DIR, 'best_asl_model.pkl')
os.makedirs('models', exist_ok=True)

# Skip model training if model is already saved in Google Drive
if os.path.exists(drive_model_path):
    print("Trained model found in Google Drive! Loading cached model...")
    rf = joblib.load(drive_model_path)
    joblib.dump(rf, 'models/best_asl_model.pkl')
    print("Model loaded successfully. Ready for evaluation or inference.")
else:
    df = pd.read_csv('data/processed/landmarks_normalized.csv')
    class_counts = df['class'].value_counts()
    df = df[df['class'].isin(class_counts[class_counts >= 2].index)]
    X = df.drop(['class', 'image_path'], axis=1)
    y = df['class']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    print("Training Random Forest...")
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_preds = rf.predict(X_test)

    print("Accuracy:", accuracy_score(y_test, rf_preds))
    print("\nClassification Report:\n", classification_report(y_test, rf_preds))

    # Save locally and back up to Google Drive
    joblib.dump(rf, 'models/best_asl_model.pkl')
    joblib.dump(rf, drive_model_path)
    print(f"Model training complete. Backed up model to Google Drive: {drive_model_path}")

Training Random Forest...
Accuracy: 0.9863950927964769

Classification Report:
               precision    recall  f1-score   support

           A       0.99      0.98      0.99       436
           B       1.00      1.00      1.00       440
           C       0.99      0.98      0.99       392
           D       0.98      1.00      0.99       491
           E       0.98      0.99      0.98       461
           F       0.99      0.99      0.99       573
           G       0.99      0.99      0.99       486
           H       0.99      0.99      0.99       475
           I       0.99      0.98      0.98       476
           J       0.99      0.99      0.99       514
           K       1.00      0.99      0.99       540
           L       1.00      0.99      1.00       504
           M       0.92      0.96      0.94       320
           N       0.95      0.93      0.94       255
           O       0.99      0.98      0.99       452
           P       0.99      0.99      0.99       408
 

## 5. Download the Model
Run this cell to download the trained model to your computer.

In [ ]:
import os

drive_model_path = os.path.join(DRIVE_DIR, 'best_asl_model.pkl')
if os.path.exists(drive_model_path):
    print(f"Your model is safely stored in Google Drive at: {drive_model_path}")
else:
    print("Model file not found. Make sure to run the training step first.")

Your model is safely stored in Google Drive at: /content/drive/MyDrive/asl_project_files/best_asl_model.pkl
